# Residência em IA — Rodada Pós-Gate 9 (Multi-Seed & Novos Modelos)

Notebook preparado para execução em **Google Colab (A100 80GB / G6 80+ GB)** com:
- Retomada automática via `resume_journal.py` salva diretamente no **Google Drive**;
- Blocos: `smoke` (24 itens), `bloco_a` (Gate 9 multi-seed), `bloco_b` (Llama 3.1 8B, Ministral 3 8B, Qwen3 8B, Gemma 4 E4B) e `bloco_c` (Llama 3.3 70B INT8).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # Configure HF_TOKEN nos Secrets do Colab
OUTPUT_ROOT = '/content/drive/MyDrive/residencia_BAP_colab/results_colab'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Clonar ou atualizar repositório
REPO_DIR = '/content/residencia_run1_BAP'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/motafernando/residencia_run1_BAP.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
# Instalar dependências atualizadas compatíveis com Llama 3.1/3.3, Ministral 3, Qwen3 e Gemma 4
!pip install -q -U transformers accelerate "huggingface-hub<2.0" bitsandbytes safetensors sentencepiece "protobuf<7.0.0" psutil

In [ ]:
# 1) Smoke test obrigatório (24 itens) para validar templates, enable_thinking=False e VRAM do 70B INT8
!python colab/run_colab_grid.py --block smoke --output-root {OUTPUT_ROOT}

In [ ]:
# 2) Bloco A — Multi-seed das runs mais óbvias do Gate 9 (seeds 2027-2030)
!python colab/run_colab_grid.py --block bloco_a --output-root {OUTPUT_ROOT}

In [ ]:
# 3) Bloco B — Últimos pesos e modelos equivalentes (~8B)
# Dica: você pode filtrar por modelo, ex: --filter qwen3_8b
!python colab/run_colab_grid.py --block bloco_b --output-root {OUTPUT_ROOT}

In [ ]:
# 4) Bloco C — Llama 3.3 70B INT8 (requer 80+ GB VRAM)
!python colab/run_colab_grid.py --block bloco_c --output-root {OUTPUT_ROOT}

In [ ]:
# 5) Consolidação final com parser v15 + métricas multi-seed + taxa de inversão por item
!python analises/consolidar_multiseed.py --results-root {OUTPUT_ROOT} --output-dir {OUTPUT_ROOT}/consolidado_v15